# 01 data exploration

Code-only reproduction copy. Outputs and execution counts have been cleared. Configure the lab settings and provide the documented input data before running. Original research evidence remains outside this folder.


In [ ]:
# Keep all inputs and outputs inside this reproduction copy.
from pathlib import Path
import sys
import re

_here = Path.cwd().resolve()
PUBLICATION_ROOT = next((p for p in (_here, _here.parent)
    if (p / '.publication-root').is_file()), None)
if PUBLICATION_ROOT is None:
    raise FileNotFoundError('Start the kernel in this repository root or its notebooks directory.')
if str(PUBLICATION_ROOT) not in sys.path:
    sys.path.insert(0, str(PUBLICATION_ROOT))
from src.lab_config import load_lab_config
LAB_CONFIG = load_lab_config()


In [ ]:
from pathlib import Path
import pandas as pd

# Repository root, assuming the notebook is in notebooks/
PROJECT_ROOT = PUBLICATION_ROOT

DATA_DIR = PROJECT_ROOT / "data" / "ground_truth"

schedule = pd.read_csv(DATA_DIR / "dataset_schedule_v3.csv")
ledger = pd.read_csv(DATA_DIR / "execution_ledger.csv")

print("Scheduled sessions:", len(schedule))
print("Ledger records:", len(ledger))

display(schedule.head())
display(ledger.head())

In [ ]:
# Examine the columns in both datasets
print("SCHEDULE COLUMNS:")
print(schedule.columns.tolist())

print("\nLEDGER COLUMNS:")
print(ledger.columns.tolist())

# Check the execution status distribution
print("\nSCHEDULE STATUS:")
print(schedule["ExecutionStatus"].value_counts())

print("\nLEDGER STATUS:")
print(ledger["Status"].value_counts())

# Check unique session IDs
print("\nUnique scheduled sessions:", schedule["SessionID"].nunique())
print("Unique ledger sessions:", ledger["SessionID"].nunique())

In [ ]:
# Get the final ledger entry for each session
final_ledger = ledger.drop_duplicates(
    subset="SessionID",
    keep="last"
).copy()

print("Final session records:", len(final_ledger))

print("\nFinal execution statuses:")
print(final_ledger["Status"].value_counts())

display(final_ledger.head())

In [ ]:
# Keep the columns we need from the final ledger
execution_results = final_ledger[
    ["SessionID", "Status", "ExecutionUTC", "Detail"]
].copy()

# Merge the final execution results with the schedule
sessions = schedule.merge(
    execution_results,
    on="SessionID",
    how="left",
    validate="one_to_one"
)

# Check the merged dataset
print("Total merged sessions:", len(sessions))

print("\nFinal execution statuses:")
print(sessions["Status"].value_counts(dropna=False))

display(
    sessions[
        ["SessionID", "SimulationDay", "ScenarioID",
         "Username", "IntendedLabel", "Status"]
    ].head(10)
)

In [ ]:
import pandas as pd

# Create a separate column for the ground-truth label
sessions["GroundTruthLabel"] = sessions["IntendedLabel"].where(
    sessions["Status"] == "Completed",
    pd.NA
)
sessions["GroundTruthLabel"] = sessions["GroundTruthLabel"].astype("Int64")

# Count the labels
print("Ground-truth label distribution:")
print(
    sessions["GroundTruthLabel"]
    .value_counts(dropna=False)
)

# Count malicious scenarios that completed
print("\nCompleted malicious sessions by scenario:")
print(
    sessions[
        (sessions["Status"] == "Completed") &
        (sessions["GroundTruthLabel"] == 1)
    ]["ScenarioID"].value_counts().sort_index()
)

# Examine failed sessions
print("\nFailed sessions:")
display(
    sessions.loc[
        sessions["Status"] == "Failed",
        ["SessionID", "ScenarioID", "IntendedLabel",
         "GroundTruthLabel", "Status"]
    ]
)

In [ ]:
import matplotlib.pyplot as plt

# Keep completed sessions with assigned labels
completed_sessions = sessions[
    sessions["GroundTruthLabel"].notna()
].copy()

# Count benign and malicious sessions
class_counts = completed_sessions["GroundTruthLabel"].value_counts().sort_index()

# Calculate percentages
class_percentages = (
    class_counts / class_counts.sum() * 100
).round(2)

print("Class distribution:")
print(class_counts)

print("\nClass percentages:")
print(class_percentages)

# Plot the distribution
plt.figure(figsize=(7, 4))
plt.bar(["Benign", "Malicious"], class_counts.values)
plt.title("Ground-Truth Class Distribution")
plt.xlabel("Session Class")
plt.ylabel("Number of Sessions")
plt.show()

In [ ]:
# Select completed malicious sessions
malicious_sessions = completed_sessions[
    completed_sessions["GroundTruthLabel"] == 1
].copy()

# Count sessions for each malicious scenario
scenario_counts = (
    malicious_sessions["ScenarioID"]
    .value_counts()
    .sort_index()
)

print("Malicious scenario distribution:")
print(scenario_counts)

# Visualise the scenario distribution
plt.figure(figsize=(8, 4))
plt.bar(scenario_counts.index, scenario_counts.values)

plt.title("Distribution of Malicious Scenarios")
plt.xlabel("Malicious Scenario")
plt.ylabel("Number of Sessions")

plt.show()

In [ ]:
# Count completed benign and malicious sessions for each day
daily_distribution = pd.crosstab(
    completed_sessions["SimulationDay"],
    completed_sessions["GroundTruthLabel"]
)

# Give the columns meaningful names
daily_distribution = daily_distribution.rename(
    columns={0: "Benign", 1: "Malicious"}
)

# Ensure days appear in numerical order
daily_distribution.index = daily_distribution.index.astype(int)
daily_distribution = daily_distribution.sort_index()

# Calculate the total sessions for each day
daily_distribution["Total"] = daily_distribution.sum(axis=1)

display(daily_distribution)